# OpenPlaque v0.4 — Standalone Boundary-Refinement Parameter Tuning

This notebook starts from a fresh Colab/runtime, loads the OpenPlaque code, runs segmentation if needed, tunes boundary-refinement parameters, and writes CSV/JSON/HTML outputs.

Research use only. Not for clinical decision-making.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Fresh-runtime setup
# 1) Clone/update the main OpenPlaque repository for study loading + segmentation.
# 2) Overlay the v0.4 working tuning code from this ZIP if it is available.
#
# In Colab, upload OpenPlaque_v0_4_Working_Boundary_Tuning.zip to /content
# or put it in /content/drive/MyDrive/OpenPlaque/.

from pathlib import Path
import os, sys, shutil, zipfile, subprocess

REPO_DIR = Path('/content/OpenPlaque')
ZIP_CANDIDATES = [
    Path('/content/OpenPlaque_v0_4_Working_Boundary_Tuning.zip'),
    Path('/content/drive/MyDrive/OpenPlaque/OpenPlaque_v0_4_Working_Boundary_Tuning.zip'),
]
EXTRACT_DIR = Path('/content/OpenPlaque_v0_4_Working_Boundary_Tuning')

if not REPO_DIR.exists():
    !git clone https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
else:
    !git -C /content/OpenPlaque pull

zip_path = next((p for p in ZIP_CANDIDATES if p.exists()), None)
if zip_path is not None:
    print('Using v0.4 ZIP:', zip_path)
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(EXTRACT_DIR)
    # The ZIP may contain files at root or inside openplaque_v04.
    possible_srcs = [EXTRACT_DIR / 'src' / 'openplaque', EXTRACT_DIR / 'openplaque_v04' / 'src' / 'openplaque']
    overlay_src = next((p for p in possible_srcs if p.exists()), None)
    if overlay_src is None:
        raise FileNotFoundError('Could not find src/openplaque inside the v0.4 ZIP.')
    target_src = REPO_DIR / 'src' / 'openplaque'
    target_src.mkdir(parents=True, exist_ok=True)
    for py in overlay_src.glob('*.py'):
        shutil.copy2(py, target_src / py.name)
        print('Overlayed', py.name)
else:
    print('No v0.4 ZIP found. Using repository code only; tuning.py must already be present in the repo.')

SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('OpenPlaque source:', SRC_DIR)

In [ ]:
# Install requirements. Rerun this cell after switching runtime type to GPU.
!wget -q https://raw.githubusercontent.com/pazzani/OpenPlaque/main/requirements-colab.txt -O /content/requirements-colab.txt
!pip install -q -r /content/requirements-colab.txt
!pip install -q pandas scipy matplotlib SimpleITK pydicom nbformat

# GPU check
!nvidia-smi

In [ ]:
# Configure nnU-Net environment
import os
from pathlib import Path

os.environ['nnUNet_raw'] = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results'] = '/content/nnUNet_results'

for d in [os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed'], os.environ['nnUNet_results']]:
    Path(d).mkdir(parents=True, exist_ok=True)

print('nnUNet_results:', os.environ['nnUNet_results'])

In [ ]:
# Extract nnU-Net model weights if needed
from pathlib import Path
import zipfile

model_zip = Path('/content/drive/MyDrive/OpenPlaque/models/Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip')
model_target = Path('/content/nnUNet_results/Dataset001_CCTA_DHM')

if model_target.exists():
    print('Model already extracted:', model_target)
else:
    if not model_zip.exists():
        raise FileNotFoundError(f'Missing model ZIP: {model_zip}')
    with zipfile.ZipFile(model_zip) as z:
        z.extractall('/content/nnUNet_results')
    print('Extracted model to /content/nnUNet_results')

!find /content/nnUNet_results -maxdepth 4 | head -80

In [ ]:
# Load the full DICOM study
from openplaque.study import OpenPlaqueStudy

study_zip = '/content/drive/MyDrive/OpenPlaque/Full_DICOM.zip'
study = OpenPlaqueStudy(study_zip)
study.summary()

In [ ]:
# Auto-detect artery series, with UCLA fallback if detection is uncertain
from openplaque.artery_detection import detect_artery_series

try:
    detected = detect_artery_series(study)
    print('Detected artery series:', detected)
except Exception as e:
    print('Auto-detection failed:', repr(e))
    detected = {'RCA': 1035, 'LCX': 1039, 'LAD': 1043}
    print('Using UCLA fallback:', detected)

# Normalize if detector returns objects rather than raw integers.
def series_number(x):
    if isinstance(x, int):
        return x
    for attr in ['series_number', 'series', 'id']:
        if hasattr(x, attr):
            return int(getattr(x, attr))
    if isinstance(x, dict):
        for key in ['series_number', 'series', 'id']:
            if key in x:
                return int(x[key])
    return int(x)

arteries = {k: series_number(v) for k, v in detected.items()}
print('Using artery series:', arteries)

In [ ]:
# Load curved coronary reformats
image_lad, volume_lad, _ = study.load_series(arteries['LAD'])
image_rca, volume_rca, _ = study.load_series(arteries['RCA'])
image_lcx, volume_lcx, _ = study.load_series(arteries['LCX'])

print('LAD:', volume_lad.shape, image_lad.GetSpacing())
print('RCA:', volume_rca.shape, image_rca.GetSpacing())
print('LCX:', volume_lcx.shape, image_lcx.GetSpacing())

In [ ]:
# Run segmentation. This requires a GPU runtime.
from openplaque.segmentation import segment_vessel

lad_report = segment_vessel(image_lad, volume_lad, 'LAD')
lad_report.summary()
lad_report.show_overlay(label=2)

rca_report = segment_vessel(image_rca, volume_rca, 'RCA')
rca_report.summary()
rca_report.show_overlay(label=2)

lcx_report = segment_vessel(image_lcx, volume_lcx, 'LCX')
lcx_report.summary()
lcx_report.show_overlay(label=2)

reports = [lad_report, rca_report, lcx_report]

In [ ]:
# Boundary-refinement parameter tuning
from openplaque.tuning import (
    tune_boundary_parameters,
    best_parameters,
    best_rows_by_vessel,
    save_tuning_outputs,
    apply_refinement_with_params,
    plot_tuning_summary,
)

# Smaller grid for a first run. Expand after confirming the workflow.
parameter_grid = {
    'min_component_voxels': [1, 5, 10, 25, 50],
    'lumen_distance_voxels': [0, 1, 2],
    'high_hu_threshold': [None, 850, 1000],
    'low_hu_threshold': [None, -100],
    'erode_core': [False],
    'erosion_iterations': [1],
}

tuning_results = tune_boundary_parameters(reports, parameter_grid=parameter_grid)
selected_params = best_parameters(tuning_results)
best_by_vessel = best_rows_by_vessel(tuning_results)

print('Rows tested:', len(tuning_results))
print('
Selected global parameters:')
print(selected_params)

display(best_by_vessel)
display(tuning_results.head(20))

In [ ]:
# Save CSV, JSON, HTML, and plots
from pathlib import Path

output_dir = Path('/content/drive/MyDrive/OpenPlaque/Boundary_Tuning')
paths = save_tuning_outputs(tuning_results, output_dir, selected_params=selected_params)
plot_paths = plot_tuning_summary(tuning_results, output_dir)
selected_refinements = apply_refinement_with_params(reports, selected_params)

print('Saved outputs:')
for name, path in paths.items():
    print(f'- {name}: {path}')
for path in plot_paths:
    print(f'- plot: {path}')

In [ ]:
# Display best parameters later without rerunning tuning
from openplaque.tuning import best_parameters, best_rows_by_vessel

selected_params = best_parameters(tuning_results)
best_by_vessel = best_rows_by_vessel(tuning_results)

print('Best global boundary-refinement parameters')
print('-' * 55)
for k, v in selected_params.items():
    print(f'{k}: {v}')

print('
Best candidate by vessel:')
display(best_by_vessel)